In [45]:
%load_ext autoreload
%autoreload 2

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [46]:
from pathlib import Path

import numpy as np
import pandas as pd
import plotly.graph_objects as go

PLL_DIR = Path.cwd().resolve().parents[2] / "results" / "pll"  # notebook lives in sw/tools/notebooks/
RESULT_CSV = None  # e.g. PLL_DIR / "pll_spectrum_div_S5_20261005_170000.csv"; None = newest run

if RESULT_CSV is None:
    RESULT_CSV = sorted((p for p in PLL_DIR.glob("pll_spectrum_div_*.csv") if not p.stem.endswith("_traces")),
                        key=lambda p: p.stat().st_mtime)[-1]
RESULT_CSV = Path(RESULT_CSV)
TRACE_CSV = RESULT_CSV.with_name(RESULT_CSV.stem + "_traces.csv")
divs = pd.read_csv(RESULT_CSV).sort_values(["total_div", "pll_clk_o_en"]).reset_index(drop=True)
traces = pd.read_csv(TRACE_CSV) if TRACE_CSV.exists() else pd.DataFrame(columns=["divider", "kind"])
divs["path"] = np.where(divs["pll_clk_o_en"] == 1, "inside the PLL", "outside the PLL")
if "video_filter" not in divs:  # runs before 2026-10-07: only the full curve used detector/averaging, markers defaults
    for col in ("detector", "average_type"):
        divs[col] = divs[col].astype(str) + " (curve only)" if col in divs else "-"
    divs["video_filter"] = "-"
# PLL output and external divider from the divider settings (runs before these columns were written lack them)
divs["pll_div"] = 2.0 ** divs["set_div_freq"]
divs["ext_div"] = divs["total_div"] / divs["pll_div"]
divs["f_pll_out"] = divs["f_vco"] / divs["pll_div"]
divs["n_meas"] = divs["f_vco"] / divs["f_out_meas"]  # f_vco / carrier on the pad: should equal total_div
divs["divider_ok"] = (divs["n_meas"] / divs["total_div"] - 1).abs() < 1e-3
ok = divs[divs["status"] == "ok"]
r0 = divs.iloc[0]
print(f"{RESULT_CSV.name}: {len(divs)} dividers, {(divs['locked'] == 1).sum()} locked, {len(ok)} measured ok; "
      f"f_vco {r0['f_vco'] / 1e6:g} MHz, c{r0['vco_tune_coarse']:.0f} min {r0['vco_current_min']:.0f} "
      f"max {r0['vco_current_max']:.0f}, |Kvco| {r0['kvco_pred'] / 1e6:.0f} MHz/V; common band {r0['common_band']} Hz")
no_carrier = divs[divs["status"] == "no_carrier"]
if len(no_carrier):
    print("no carrier on the pad (not measured): " + ", ".join(
        f"{r.divider} ({r.f_out / 1e6:g} MHz on the pad, {r.carrier_dbm:.1f} dBm)" for r in no_carrier.itertuples()))

pd.DataFrame({
    "divider": divs["divider"],
    "N": divs["total_div"],
    "PLL output [MHz]": divs["f_pll_out"] / 1e6,
    "ext div": divs["ext_div"],
    "f_pad expected [MHz]": divs["f_out"] / 1e6,
    "f_pad [MHz]": divs["f_out_meas"] / 1e6,  # analyzer centre frequency = carrier on the pad
    "N measured": divs["n_meas"],
    "divider ok": divs["divider_ok"],
    "carrier on the pad [dBm]": divs["carrier_dbm"],
    "band up to [Hz]": divs["offset_max"],
    "jitter common, markers [ps]": divs["time_jitter_common_ps"],
    "jitter common, curve [ps]": divs["time_jitter_trace_common_ps"],
    "jitter full band, curve [ps]": divs["time_jitter_trace_ps"],
    "jitter full band, spurs [ps]": divs["time_jitter_spur_ps"],
    "detector": divs["detector"],
    "averaging": divs["average_type"],
    "video filter": divs["video_filter"],
    "locked": divs["locked"],
    "status": divs["status"],
}).round(3)

pll_spectrum_div_S5_20261007_112110.csv: 9 dividers, 9 locked, 9 measured ok; f_vco 1000 MHz, c2 min 11 max 4, |Kvco| 1146 MHz/V; common band 1000-1e+06 Hz


,divider,N,PLL output [MHz],ext div,f_pad expected [MHz],f_pad [MHz],N measured,divider ok,carrier on the pad [dBm],band up to [Hz],"jitter common, markers [ps]","jitter common, curve [ps]","jitter full band, curve [ps]","jitter full band, spurs [ps]",detector,averaging,video filter,locked,status
0,ext /2 x /4,8,500.000,4.0,125.000,125.000,8.0,True,-4.408,50000000.0,983.539,1369.185,1800.398,605.485,RMS,POW,LIN,1,ok
1,int /8,8,125.000,1.0,125.000,125.000,8.0,True,-4.385,50000000.0,1212.315,1365.860,1798.231,637.381,RMS,POW,LIN,1,ok
2,ext /1 x /10,10,1000.000,10.0,100.000,100.000,10.0,True,-2.548,10000000.0,1254.362,1268.294,1583.321,628.683,RMS,POW,LIN,1,ok
3,int /16,16,62.500,1.0,62.500,62.500,16.0,True,0.509,10000000.0,742.678,1183.607,1459.864,598.882,RMS,POW,LIN,1,ok
4,int /32,32,31.250,1.0,31.250,31.250,32.0,True,-1.864,10000000.0,682.395,1088.569,1339.401,559.018,RMS,POW,LIN,1,ok
5,ext /4 x /10,40,250.000,10.0,25.000,25.000,40.0,True,-2.322,10000000.0,744.799,1204.303,1442.044,588.475,RMS,POW,LIN,1,ok
6,int /64,64,15.625,1.0,15.625,15.625,64.0,True,-3.102,1000000.0,748.346,1037.168,1037.168,499.844,RMS,POW,LIN,1,ok
7,ext /1 x /100,100,1000.000,100.0,10.000,10.000,100.0,True,-4.024,1000000.0,587.482,1169.971,1169.971,605.006,RMS,POW,LIN,1,ok
8,int /128,128,7.812,1.0,7.812,7.812,128.0,True,-4.556,1000000.0,631.202,1101.533,1101.533,549.932,RMS,POW,LIN,1,ok


In [47]:
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import bin_phase_noise

INK, GRID = "#52514e", "#e4e3df"
PATH_COLORS = {"inside the PLL": "#2a78d6", "outside the PLL": "#eb6834"}
REF_TXT = f"f_ref (PLL input) {divs['f_ref'].iloc[0] / 1e6:.4g} MHz: reference spurs at k x f_ref"  # same for every divider
OFFSET_AXIS = dict(type="log", tickvals=[1e3, 1e4, 1e5, 1e6, 1e7, 1e8],
                   ticktext=["1 kHz", "10 kHz", "100 kHz", "1 MHz", "10 MHz", "100 MHz"],
                   minor=dict(showgrid=True, gridcolor="#f1f0ed"))
FREQ_AXIS = dict(type="log", tickvals=[1, 10, 100, 1e3], ticktext=["1 MHz", "10 MHz", "100 MHz", "1 GHz"],
                 minor=dict(showgrid=True, gridcolor="#f1f0ed"))
N_BINS = 300  # points per plotted phase-noise curve (spread over F_start-F_stop); None = every measured point


def offset_axis(f_start, f_stop, f_axis):
    """x-axis settings for an offset plot shown from f_start to f_stop [Hz]; f_axis "log" or "lin"."""
    if f_axis == "log":
        return dict(OFFSET_AXIS, range=[np.log10(f_start), np.log10(f_stop)])
    return dict(type="linear", range=[f_start, f_stop], ticksuffix="Hz", exponentformat="SI")


def in_range(df, f_start, f_stop):
    """Rows of df with the offset (column freq) between f_start and f_stop, so the y axis fits the shown band."""
    return df[(df["freq"] >= f_start) & (df["freq"] <= f_stop)]


def binned(df, f_start, f_stop, f_axis):
    """The curve in df (columns freq, level_dbm) between f_start and f_stop, averaged into N_BINS bins spaced for
    f_axis, in linear power (pll_util.bin_phase_noise): display only, the jitter uses the measured points."""
    df = in_range(df, f_start, f_stop).sort_values("freq")
    if N_BINS is None or len(df) < 2:
        return df
    if f_axis == "log":
        edges = np.geomspace(f_start, f_stop, N_BINS + 1)
    else:
        edges = np.linspace(f_start, f_stop, N_BINS + 1)
    freq, level = bin_phase_noise(df["freq"], df["level_dbm"], edges)
    return pd.DataFrame({"freq": freq, "level_dbm": level})


def save(fig, name):
    """Interactive HTML and a static PNG (kaleido) next to the CSV, then show the figure."""
    base = RESULT_CSV.with_name(f"{RESULT_CSV.stem}_{name}")
    fig.write_html(base.with_suffix(".html"))
    try:
        fig.write_image(base.with_suffix(".png"), width=1100, height=fig.layout.height or 600, scale=2)
        print(f"saved {base}.html and .png")
    except Exception as e:  # kaleido >= 1.0 needs a Chrome/Chromium install (`plotly_get_chrome`)
        print(f"saved {base}.html; PNG export failed: {e}")
    fig.show()

In [48]:
# Divider check: N measured (f_vco / carrier on the pad) against the set total divider; on the line when right.
# (A digital divider is exact: this only catches a wrong N, not a precision error.)
fig = go.Figure()
lim = [divs["total_div"].min() / 1.5, divs["total_div"].max() * 1.5]
fig.add_scatter(x=lim, y=lim, mode="lines", line=dict(color=INK, width=1, dash="dash"), hoverinfo="skip",
                name="N measured = N set")
for path, d in divs[divs["n_meas"].notna()].groupby("path"):
    fig.add_scatter(x=d["total_div"], y=d["n_meas"], mode="markers", name=path,
                    marker=dict(size=11, color=PATH_COLORS[path], line=dict(color="white", width=1)),
                    customdata=np.c_[d["divider"], d["f_out_meas"] / 1e6, d["divider_ok"]],
                    hovertemplate="%{customdata[0]}: N set %{x}, measured %{y:.4f} (carrier on the pad "
                                  "%{customdata[1]:.6f} MHz), ok %{customdata[2]}<extra></extra>")
fig.update_layout(title=f"Divider check: N measured (f_vco / carrier on the pad) vs N set, {REF_TXT}", template="plotly_white",
                  height=480, legend=dict(orientation="h", y=-0.2))
fig.update_xaxes(title="total divider set", type="log", gridcolor=GRID)
fig.update_yaxes(title="N measured", type="log", gridcolor=GRID)
save(fig, "divider_check")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_divider_check.html and .png


In [55]:
F_start, F_stop = 1e3, 10e6  # offset range shown [Hz]
f_axis = "log"  # "log" or "lin"

# Phase noise per divider, split in two comparisons. Left: referred to the VCO (pad + 20 log10 N), where the curves
# ideally coincide; right: as measured on the pad. Legend: total output divider, its split inside / outside the
# PLL, f_pad. Curves binned to N_BINS points (linear power average, display only).
from plotly.subplots import make_subplots

PALETTE_N = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#4a3aa7", "#e34948"]


def plot_phase_noise(sel, title, name):
    """Full curves (L_trace) of the dividers in `sel`: referred to the VCO (left) and on the pad (right)."""
    fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                        subplot_titles=("referred to the VCO (pad + 20 log10 N)", "measured on the pad"))
    for i, r in enumerate(sel.sort_values(["total_div", "ext_div"]).itertuples()):
        lt = traces[(traces["divider"] == r.divider) & (traces["kind"] == "L_trace")]
        lt = binned(lt, F_start, F_stop, f_axis)
        if lt.empty:
            continue
        label = (f"output /{r.total_div:.0f} (/{r.pll_div:.0f} in the PLL x /{r.ext_div:.0f} external): "
                 f"f_pad {r.f_out_meas / 1e6:.4g} MHz")
        line = dict(color=PALETTE_N[i % len(PALETTE_N)], width=2, dash="dot" if r.ext_div > 1 else "solid")
        for col, add_db in ((1, 20 * np.log10(r.total_div)), (2, 0.0)):
            fig.add_scatter(x=lt["freq"], y=lt["level_dbm"] + add_db, mode="lines", name=label, legendgroup=label,
                            showlegend=col == 1, line=line, row=1, col=col,
                            hovertemplate=f"{r.divider}<br>%{{x:.3s}}Hz: %{{y:.1f}} dBc/Hz<extra></extra>")
    fig.update_layout(title=f"{title}, {REF_TXT}<br><sup>solid: divider inside the PLL only, dotted: + external "
                            "divider</sup>",
                      template="plotly_white", height=600, legend=dict(orientation="h", y=-0.2, font=dict(size=10)),
                      hoverlabel=dict(namelength=-1))
    fig.update_xaxes(title="offset from the carrier", gridcolor=GRID, **offset_axis(F_start, F_stop, f_axis))
    fig.update_yaxes(title="L(f) [dBc/Hz]", gridcolor=GRID)
    save(fig, name)


# 1. Same total divider N, split differently between inside and outside the PLL
same_n = ok.groupby("total_div").filter(lambda d: d["pll_div"].nunique() > 1)
if len(same_n):
    plot_phase_noise(same_n, "Same total divider N, divided inside vs outside the PLL", "phase_noise_same_n")
# 2. Fixed external divider, divider inside the PLL varied: one figure per external divider
for ext, d in ok.groupby("ext_div"):
    if d["pll_div"].nunique() > 1:
        plot_phase_noise(d, f"External divider fixed at /{ext:.0f}, divider inside the PLL varied",
                         f"phase_noise_ext{ext:.0f}")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_phase_noise_same_n.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_phase_noise_ext1.html and .png


saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_phase_noise_ext10.html and .png


In [ ]:
F_start, F_stop = None, None  # offset from the carrier shown [Hz] (may be negative); None = edge of the trace
f_axis = "lin"  # "lin" or "log" (log: upper sideband only)

# Averaged wide spectrum around the carrier per divider: level relative to the carrier (trace maximum), in the trace's
# RBW (one sweep, not binned). Left: referred to the VCO (+ 20 log10 N), where the sidebands of every divider ideally
# coincide; the carrier itself (within 10 RBW) is left out there, +20 log10 N only applies to the noise sidebands.
# Right: as measured on the pad.
from plotly.subplots import make_subplots

fig = make_subplots(rows=1, cols=2, horizontal_spacing=0.08,
                    subplot_titles=("referred to the VCO (pad + 20 log10 N, carrier left out)", "measured on the pad"))
for i, r in enumerate(ok.sort_values(["total_div", "ext_div"]).itertuples()):
    d = traces[(traces["divider"] == r.divider) & (traces["kind"] == "spectrum")]
    if d.empty:
        continue
    d = d.assign(freq=d["freq"] - d["center"], level_dbc=d["level_dbm"] - d["level_dbm"].max())  # offset, rel. to carrier
    lo = d["freq"].min() if F_start is None else F_start
    hi = d["freq"].max() if F_stop is None else F_stop
    if f_axis == "log":
        lo = max(lo, d.loc[d["freq"] > 0, "freq"].min())
    d = in_range(d, lo, hi)
    sidebands = d[d["freq"].abs() >= 10 * d["rbw"]]
    sidebands = pd.concat([sidebands[sidebands["freq"] < 0], pd.DataFrame({"freq": [0.0], "level_dbc": [np.nan]}),
                           sidebands[sidebands["freq"] > 0]])  # gap at the carrier
    label = f"{r.divider}: f_pad {r.f_out_meas / 1e6:.4g} MHz"
    line = dict(color=PALETTE_N[i % len(PALETTE_N)], width=1.5, dash="dot" if r.ext_div > 1 else "solid")
    for col, curve, add_db, where in ((1, sidebands, 20 * np.log10(r.total_div), "at the VCO"), (2, d, 0.0, "on the pad")):
        fig.add_scatter(x=curve["freq"], y=curve["level_dbc"] + add_db, mode="lines", name=label, legendgroup=label,
                        showlegend=col == 1, line=line, row=1, col=col,
                        hovertemplate=f"{r.divider}<br>%{{x:.4s}}Hz: %{{y:.1f}} dBc {where}<extra></extra>")
rbw = traces.loc[traces["kind"] == "spectrum", "rbw"]
fig.update_layout(title=f"Spectrum around the carrier per divider (RBW {rbw.iloc[0]:g} Hz), {REF_TXT}<br>"
                        "<sup>solid: divider inside the PLL only, dotted: + external divider</sup>" if len(rbw)
                  else f"Spectrum around the carrier per divider, {REF_TXT}",
                  template="plotly_white", height=600, legend=dict(orientation="h", y=-0.2, font=dict(size=10)),
                  hoverlabel=dict(namelength=-1))
if f_axis == "log":
    fig.update_xaxes(**OFFSET_AXIS)
else:
    fig.update_xaxes(type="linear", ticksuffix="Hz", exponentformat="SI")
fig.update_xaxes(title="offset from the carrier", gridcolor=GRID)
fig.update_yaxes(title="level relative to the carrier [dBc in RBW]", gridcolor=GRID)
save(fig, "spectrum")

In [51]:
# RMS jitter over the common band per divider (markers and full curve): should be the same for every divider.
band = ok["common_band"].iloc[0] if len(ok) else ""
fig = go.Figure()
for col, name, color in (("time_jitter_trace_common_ps", "full curve", "#2a78d6"),
                         ("time_jitter_common_ps", "markers", "#eb6834")):
    fig.add_bar(x=ok["divider"], y=ok[col], name=name, marker_color=color,
                customdata=np.c_[ok["total_div"], ok["f_out"] / 1e6, ok["carrier_dbm"]],
                hovertemplate=(f"{name}: %{{y:.1f}} ps<br>%{{x}} (/%{{customdata[0]:.0f}}, %{{customdata[1]:g}} MHz, "
                               "carrier %{customdata[2]:.1f} dBm)<extra></extra>"))
fig.update_layout(title=f"RMS time jitter over the common band ({band} Hz) per divider (same at pad, PLL output, VCO), {REF_TXT}", template="plotly_white",
                  height=470, barmode="group", legend=dict(orientation="h", y=-0.3))
fig.update_xaxes(title="divider", categoryorder="array", categoryarray=list(ok["divider"]), gridcolor=GRID)
fig.update_yaxes(title="RMS time jitter [ps]", gridcolor=GRID, rangemode="tozero")
save(fig, "jitter_common")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_jitter_common.html and .png


In [52]:
# Carrier level on the pad vs pad frequency: how well the pad drives each frequency. Colour = path.
fig = go.Figure()
for path, d in divs[divs["carrier_dbm"].notna()].groupby("path"):
    fig.add_scatter(x=d["f_out"] / 1e6, y=d["carrier_dbm"], mode="markers", name=path,
                    marker=dict(size=11, color=PATH_COLORS[path], line=dict(color="white", width=1)),
                    customdata=np.c_[d["divider"], d["total_div"], d["f_pll_out"] / 1e6],
                    hovertemplate=("%{customdata[0]} (/%{customdata[1]}): pad %{x:g} MHz, %{y:.1f} dBm "
                                   "(PLL output %{customdata[2]} MHz)<extra></extra>"))
fig.update_layout(title=f"Carrier level on the pad vs pad frequency, {REF_TXT}", template="plotly_white", height=450,
                  legend=dict(orientation="h", y=-0.25))
fig.update_xaxes(title="pad frequency (after the external divider)", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="carrier [dBm]", gridcolor=GRID)
save(fig, "carrier_vs_f_out")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_carrier_vs_f_out.html and .png


In [53]:
# L(f) at 1 kHz referred to the VCO vs pad frequency: the same VCO behind every divider, so this should be flat as
# long as the divider and the pad add less noise than the divided VCO. Colour = path.
fig = go.Figure()
for path, d in divs[divs["L_vco_1000"].notna()].groupby("path"):
    fig.add_scatter(x=d["f_out_meas"] / 1e6, y=d["L_vco_1000"], mode="markers", name=path,
                    marker=dict(size=11, color=PATH_COLORS[path], line=dict(color="white", width=1)),
                    customdata=np.c_[d["divider"], d["total_div"], d["L_out_1000"]],
                    hovertemplate=("%{customdata[0]} (/%{customdata[1]}): pad %{x:.4g} MHz, %{y:.1f} dBc/Hz at the VCO "
                                   "(%{customdata[2]:.1f} dBc/Hz on the pad)<extra></extra>"))
fig.update_layout(title=f"L(1 kHz) referred to the VCO vs pad frequency, {REF_TXT}", template="plotly_white",
                  height=450, legend=dict(orientation="h", y=-0.25))
fig.update_xaxes(title="f_pad (analyzer centre frequency)", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="L(1 kHz) referred to the VCO [dBc/Hz]", gridcolor=GRID)
save(fig, "L_vco_1k_vs_f_pad")

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_L_vco_1k_vs_f_pad.html and .png


In [54]:
# RMS jitter over the common band from the full curves. The time jitter from a divider's pad curve and from its own
# VCO-referred curve (L + 20 log10 N, integrated at N x f_pad) are identical by construction; the real check is each
# divider against one reference VCO curve: the divider with the lowest f_pad (least excess close-in noise). An ideal
# divider keeps the time jitter, so every pad should show the reference time jitter; excess = quadrature difference.
import sys

sys.path.insert(0, str(PLL_DIR.parents[1]))  # repo root
from sw.lib.pll_util import integrate_phase_noise

lo, hi = (float(x) for x in ok["common_band"].iloc[0].split("-"))


def band_curve(divider):
    """(offsets, L on the pad) of the divider's full curve inside the common band."""
    lt = traces[(traces["divider"] == divider) & (traces["kind"] == "L_trace")]
    lt = lt[(lt["freq"] >= lo) & (lt["freq"] <= hi)].sort_values("freq")
    return lt["freq"].to_numpy(), lt["level_dbm"].to_numpy()


ref = ok.loc[ok["f_out_meas"].idxmin()]
o_ref, l_ref = band_curve(ref["divider"])
t_ref, phi_ref = integrate_phase_noise(ref["f_out_meas"] * ref["total_div"], o_ref,
                                       l_ref + 20 * np.log10(ref["total_div"]))  # VCO-referred reference

rows = []
for r in ok.itertuples():
    o, l = band_curve(r.divider)
    if len(o) < 2:
        continue
    t_pad, phi_pad = integrate_phase_noise(r.f_out_meas, o, l)
    t_vco, phi_vco = integrate_phase_noise(r.f_out_meas * r.total_div, o, l + 20 * np.log10(r.total_div))
    rows.append({"divider": r.divider, "N": r.total_div, "f_pad [MHz]": r.f_out_meas / 1e6,
                 "jitter pad curve [ps]": t_pad * 1e12, "jitter own VCO curve [ps]": t_vco * 1e12,
                 "jitter reference VCO curve [ps]": t_ref * 1e12,
                 "excess over reference [ps]": np.sqrt(max(t_pad ** 2 - t_ref ** 2, 0)) * 1e12,
                 "phase jitter pad [mrad]": phi_pad * 1e3, "phase jitter VCO [mrad]": phi_vco * 1e3,
                 "phase jitter pad from reference [mrad]": phi_ref / r.total_div * 1e3})
jit = pd.DataFrame(rows).sort_values("f_pad [MHz]").reset_index(drop=True)

fig = go.Figure()
fig.add_scatter(x=jit["f_pad [MHz]"], y=jit["jitter pad curve [ps]"], mode="markers", name="from the pad curve",
                marker=dict(size=11, color="#2a78d6", line=dict(color="white", width=1)), customdata=jit[["divider"]],
                hovertemplate="%{customdata[0]}: f_pad %{x:.4g} MHz, %{y:.2f} ps<extra></extra>")
fig.add_hline(y=t_ref * 1e12, line=dict(color="#eb6834", dash="dash"),
              annotation_text=f"reference VCO curve ({ref['divider']}): {t_ref * 1e12:.2f} ps",
              annotation_position="top left")
fig.update_layout(title=f"RMS time jitter over {lo:g}-{hi:g} Hz: pad curves vs the reference VCO curve, {REF_TXT}",
                  template="plotly_white", height=450, legend=dict(orientation="h", y=-0.25))
fig.update_xaxes(title="f_pad (analyzer centre frequency)", gridcolor=GRID, **FREQ_AXIS)
fig.update_yaxes(title="RMS time jitter [ps]", gridcolor=GRID, rangemode="tozero")
save(fig, "jitter_pad_vs_vco")
jit.round(3)

saved /users/micas/wvandest/projects/lagd-meas/results/pll/pll_spectrum_div_S5_20261007_112110_jitter_pad_vs_vco.html and .png


,divider,N,f_pad [MHz],jitter pad curve [ps],jitter own VCO curve [ps],jitter reference VCO curve [ps],excess over reference [ps],phase jitter pad [mrad],phase jitter VCO [mrad],phase jitter pad from reference [mrad]
0,int /128,128,7.812,1101.533,1101.533,1101.533,0.000,54.071,6921.129,54.071
1,ext /1 x /100,100,10.000,1169.971,1169.971,1101.533,394.282,73.511,7351.140,69.211
2,int /64,64,15.625,1037.168,1037.168,1101.533,0.000,101.824,6516.717,108.143
3,ext /4 x /10,40,25.000,1204.303,1204.303,1101.533,486.796,189.171,7566.852,173.028
4,int /32,32,31.250,1088.569,1088.569,1101.533,0.000,213.740,6839.677,216.285
5,int /16,16,62.500,1183.607,1183.607,1101.533,433.073,464.801,7436.819,432.571
6,ext /1 x /10,10,100.000,1268.294,1268.294,1101.533,628.645,796.892,7968.919,692.113
7,ext /2 x /4,8,125.000,1369.185,1369.185,1101.533,813.199,1075.354,8602.835,865.141
8,int /8,8,125.000,1365.860,1365.860,1101.533,807.588,1072.743,8581.945,865.141
